# Hybrid Retrieval for RAG

A retrieval component built from scratch: lexical search (BM25), semantic search (bi-encoder) and reranking (cross-encoder), combined into one pipeline over 50k passages of Polish Wikipedia, and finally wired to a generator that cites its sources.

In [ ]:
!pip install -U sentence-transformers rank_bm25 datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 30.2 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


## Three retrieval methods on a small corpus

### BM25 - lexical search

Keyword matching, with no understanding of meaning.

In [ ]:
# BM25 SEARCH - lexical search
from rank_bm25 import BM25Okapi
import re
import torch

# Corpus of example passages
corpus = [
    "Trzeba jeść dużo warzyw, owoców i uprawiać sport.",
    "Trzeba pić alkohol, imprezować i palić.",
    "Kampania polityczna bardzo napawa optymizmem, same mądre głowy chcą rządzić Polakami.",
    "Historia Polski to wiele wieków walki o niepodległość.",
    "Przemysł motoryzacyjny od lat walczy z rosnącymi kosztami produkcji.",
]
bm25_chunks = [[word.lower() for word in re.findall(r'\w+', chunk)] for chunk in corpus]
print(bm25_chunks)
bm25 = BM25Okapi(bm25_chunks)

query = "Czy lubisz warzywa i sport?"
query_chunks = [word.lower() for word in re.findall(r'\w+', query)]
print(query_chunks)

bm25_query_scores = bm25.get_scores(query_chunks)
print(bm25_query_scores)

_, top_chunks_indices_bm25 = torch.topk(torch.tensor(bm25_query_scores), k=2)
top_chunks_indices_bm25 = top_chunks_indices_bm25.tolist()
print(top_chunks_indices_bm25)

[['trzeba', 'jeść', 'dużo', 'warzyw', 'owoców', 'i', 'uprawiać', 'sport'], ['trzeba', 'pić', 'alkohol', 'imprezować', 'i', 'palić'], ['kampania', 'polityczna', 'bardzo', 'napawa', 'optymizmem', 'same', 'mądre', 'głowy', 'chcą', 'rządzić', 'polakami'], ['historia', 'polski', 'to', 'wiele', 'wieków', 'walki', 'o', 'niepodległość'], ['przemysł', 'motoryzacyjny', 'od', 'lat', 'walczy', 'z', 'rosnącymi', 'kosztami', 'produkcji']]
['czy', 'lubisz', 'warzywa', 'i', 'sport']
[1.46650973 0.38611568 0.         0.         0.        ]
[0, 1]


### Bi-encoder - semantic search

Query and documents encoded independently; similarity computed in vector space.

In [ ]:
# BI-ENCODER SEARCH - semantic search
from sentence_transformers import SentenceTransformer

bi_encoder = SentenceTransformer("multi-qa-mpnet-base-dot-v1")
chunks_embeddings = bi_encoder.encode(corpus, convert_to_tensor=True, show_progress_bar=True)
query_embedding = bi_encoder.encode(query, convert_to_tensor=True)

bi_encoder_query_scores = bi_encoder.similarity(query_embedding, chunks_embeddings)
print(bi_encoder_query_scores)

_, top_chunks_indices_bi_encoder = torch.topk(bi_encoder_query_scores, k=2)
top_chunks_indices_bi_encoder = top_chunks_indices_bi_encoder.tolist()[0]
print(top_chunks_indices_bi_encoder)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

... (6 outputs left out)

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tensor([[26.4896, 22.2884, 21.0314, 20.1221, 21.0577]], device='cuda:0')
[0, 1]


### Cross-encoder - reranking

The (query, document) pair is fed in together - more accurate, but too expensive for the whole corpus, so it only ranks the candidates produced by the two previous methods.

In [ ]:
# CROSS-ENCODER - reranking
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

reranking_indices = list(set(top_chunks_indices_bm25 + top_chunks_indices_bi_encoder))

reranking_pair = [[query, corpus[indx]] for indx in reranking_indices]
reranking_scores = reranker.predict(reranking_pair)
print(reranking_scores)

_, top_chunks_indices_reranking = torch.topk(torch.tensor(reranking_scores), k=1)
top_chunks_indices_reranking = top_chunks_indices_reranking.tolist()
print(top_chunks_indices_reranking)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

[  5.7812185 -11.227265 ]
[0]


## Corpus - Polish Wikipedia

Documents are split into overlapping 200-word passages with a 40-word overlap, so that a sentence cut at a passage boundary does not disappear from the index.

In [ ]:
from datasets import load_dataset

wikipedia_dataset = 'chrisociepa/wikipedia-pl-20230401'
dataset = load_dataset(wikipedia_dataset, split="train", streaming=True)
print(next(iter(dataset)))

README.md: 0.00B [00:00, ?B/s]

{'id': '2', 'url': 'https://pl.wikipedia.org/wiki/AWK', 'title': 'AWK', 'text': 'AWK – interpretowany język programowania, którego główną funkcją jest wyszukiwanie i przetwarzanie wzorców w plikach lub strumieniach danych. Jest także nazwą programu początkowo dostępnego dla systemów operacyjnych będących pochodnymi UNIX-a, obecnie także na inne platformy.\n\nAWK jest językiem, który w znacznym stopniu wykorzystuje tablice asocjacyjne, stringi i wyrażenia regularne. Nazwa języka pochodzi od pierwszych liter nazwisk jego autorów Alfreda V. Aho, Petera Weinbergera i Briana Kernighana. Bywa zapisywana małymi literami, odczytywana jako jedno słowo awk, wymawiana jak pierwsza sylaba w awkward.\n\nDefinicja języka AWK jest zawarta w POSIX 1003.2 Command Language And Utilities Standard. Wersja ta jest z kolei oparta na opisie z The AWK Programming Language napisanym przez Aho, Weinbergera i Kernighana, z dodatkowymi właściwościami zdefiniowanymi w wersji awk z SVR4.\n\nW wierszu poleceń podaje

In [ ]:
def build_rag_dataset(dataset, chunk_size=200, overlap_size=40, max_documents=50000):
  chunks = set()
  step = chunk_size - overlap_size

  for doc in dataset:
    if len(chunks) >= max_documents:
      return list(chunks)

    text = doc.get('text')
    if text is not None:
      words = text.split()

      if len(words) < chunk_size:
        chunks.add(text)
      else:
        doc_chunks = [" ".join(words[i: i+chunk_size]) for i in range(0, len(words), step)]
        for chunk in doc_chunks:
          chunks.add(chunk)

  return list(chunks)

In [ ]:
chunks = build_rag_dataset(dataset)
print(f"{len(chunks)}")

50009


In [ ]:
chunks[10]

'(3291) Dunlap – planetoida z grupy pasa głównego asteroid okrążająca Słońce w ciągu 5 lat i 213 dni w średniej odległości 3,15 j.a. Została odkryta 14 listopada 1982 w obserwatorium w Kiso Station przez Hirokiego Kōsai i Kiichirō Furukawę. Nazwa planetoidy pochodzi od Larry’ego Dunlapa, asystenta w Lunar and Planetary Laboratory. Przed nadaniem nazwy planetoida nosiła oznaczenie tymczasowe (3291) 1982 VX3.\n\nZobacz też \n lista planetoid 3001–4000\n lista ponumerowanych planetoid\n\nLinki zewnętrzne \n \n \n \n\nPlanetoidy pasa głównego\nNazwane planetoidy\nObiekty astronomiczne odkryte w 1982'

## Lexical index

Tokenisation with a Polish stop-word list.

In [ ]:
import re
from rank_bm25 import BM25Okapi


STOP_WORDS = {
    'ach', 'aj', 'albo', 'bardzo', 'bez', 'bo', 'być', 'ci', 'ciebie', 'cię',
    'co', 'czy', 'daleko', 'dla', 'dlaczego', 'dlatego', 'do', 'dobrze', 'dokąd',
    'dość', 'dużo', 'dwa', 'dwaj', 'dwie', 'dwoje', 'dzisiaj', 'dziś', 'gdyby',
    'gdzie', 'go', 'ich', 'ile', 'im', 'inny', 'ja', 'jak', 'jakby', 'jaki',
    'je', 'jeden', 'jedna', 'jedno', 'jego', 'jej', 'jemu', 'jest', 'jestem',
    'jeśli', 'jeżeli', 'już', 'ją', 'każdy', 'kiedy', 'kierunku', 'kto', 'ku',
    'lub', 'ma', 'mają', 'mam', 'mi', 'mnie', 'mną', 'moi', 'moja', 'moje',
    'może', 'mu', 'my', 'mój', 'na', 'nam', 'nami', 'nas', 'nasi', 'nasz',
    'nasza', 'nasze', 'natychmiast', 'nic', 'nich', 'nie', 'niego', 'niej',
    'niemu', 'nigdy', 'nim', 'nimi', 'nią', 'niż', 'obok', 'od', 'około', 'on',
    'ona', 'one', 'oni', 'ono', 'owszem', 'po', 'pod', 'ponieważ', 'przed',
    'przedtem', 'sam', 'sama', 'się', 'skąd', 'są', 'tak', 'taki', 'tam',
    'ten', 'to', 'tobie', 'tobą', 'tu', 'tutaj', 'twoi', 'twoja', 'twoje',
    'twój', 'ty', 'wam', 'wami', 'was', 'wasi', 'wasz', 'wasza', 'wasze',
    'we', 'więc', 'wszystko', 'wtedy', 'wy', 'zawsze', 'żaden', 'że'
}
bm25_chunks = [[word.lower() for word in re.findall(r'\w+', chunk) if word not in STOP_WORDS] for chunk in chunks]
bm25_encoder = BM25Okapi(bm25_chunks)

print(f"{len(bm25_chunks)=}")
print(f"{len(bm25_chunks[0])=}")

len(bm25_chunks)=50009
len(bm25_chunks[0])=178


## Semantic search models

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

bi_encoder = SentenceTransformer("multi-qa-mpnet-base-dot-v1")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/multi-qa-mpnet-base-dot-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
tokenized_chunks_corpus = bi_encoder.encode(chunks, convert_to_tensor=True, show_progress_bar=True)

Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

In [ ]:
print(tokenized_chunks_corpus.shape)
type(tokenized_chunks_corpus)

torch.Size([50009, 768])


torch.Tensor

## Hybrid retrieval

Candidates from BM25 and from the bi-encoder go into a shared pool, which the cross-encoder reranks before returning the top *N*.

In [ ]:
import torch

def search(query, top_n=3):
  print("Input question:", query)
  candidates_pool_size = top_n * 5

  # Lexical search
  bm25_query = [word.lower() for word in re.findall(r'\w+', query) if word not in STOP_WORDS]
  bm25_query_scores = bm25_encoder.get_scores(bm25_query)
  _, top_chunks_indices_bm25 = torch.topk(torch.tensor(bm25_query_scores), k=candidates_pool_size)
  top_chunks_indices_bm25 = top_chunks_indices_bm25.tolist()
  print(f"{top_chunks_indices_bm25=}")

  # Semantic search
  bi_encoder_query = bi_encoder.encode(query, convert_to_tensor=True)
  bi_encoder_query_scores = bi_encoder.similarity(bi_encoder_query, tokenized_chunks_corpus)
  _, top_chunks_indices_bi_encoder = torch.topk(bi_encoder_query_scores, k=candidates_pool_size)
  top_chunks_indices_bi_encoder = top_chunks_indices_bi_encoder.tolist()[0]
  print(f"{top_chunks_indices_bi_encoder=}")

  # Reranking
  reranking_indices = list(set(top_chunks_indices_bm25 + top_chunks_indices_bi_encoder))
  reranking_pairs = [[query, chunks[indx]] for indx in reranking_indices]
  reranking_scores = cross_encoder.predict(reranking_pairs)

  _, top_chunks_indices_reranking = torch.topk(torch.tensor(reranking_scores), k=top_n)
  top_chunks_indices_reranking = top_chunks_indices_reranking.tolist()
  print(f"{reranking_indices=}")

  chunk_indices = [reranking_indices[idx] for idx in top_chunks_indices_reranking]
  print(chunk_indices)

  retrieved_chunks = [chunks[indx] for indx in chunk_indices]

  return retrieved_chunks

In [ ]:
docs = search("Kim był Gerald z Rivii")
print(docs[0])

Input question: Kim był Gerald z Rivii
top_chunks_indices_bm25=[24961, 45720, 41921, 2226, 21388, 20416, 47993, 12872, 2405, 38984, 4950, 28256, 17922, 38308, 47418]
top_chunks_indices_bi_encoder=[16575, 12644, 46783, 37153, 12130, 15777, 19738, 46347, 35486, 29400, 16051, 27288, 45003, 2079, 22366]
reranking_indices=[24961, 17922, 46347, 21388, 45720, 27288, 19738, 35486, 2079, 37153, 15777, 38308, 2226, 16051, 47418, 46783, 20416, 41921, 16575, 12872, 38984, 45003, 4950, 29400, 22366, 28256, 12130, 12644, 2405, 47993]
[41921, 38308, 21388]
Gerald Campion (ur. 23 kwietnia 1921, zm. 9 lipca 2002) - angielski aktor najbardziej znany ze swoich ról jako Billy Bunter w telewizyjnej adaptacji książek Franka Richardsa w latach 50. XX wieku.

Ojcem Geralda był scenarzysta Cyril Campion. Gerald wystąpił w licznych filmach i programach telewizyjnych (głównie komediowych), jednak rola Buntera okazała się jego największym sukcesem.

Po rzuceniu aktorstwa prowadził restaurację.

Linki zewnętrzne 


In [ ]:
chunks[11336]

'zejściem gracza zmienianego. Jeżeli drużyna pozostanie z mniej niż trzema graczami zdolnymi do gry (w związku z akcjami dyscyplinarnymi), zostanie uznana za drużynę przegraną niezależnie od wyniku gry sprzed momentu, kiedy gra została zatrzymana. W tym wypadku zarządzony będzie wynik meczu 10:0 dla drużyny wygrywającej. Natomiast przy dostaniu kartki czerwonej zawodnik musi opuścić boisko do końca meczu. Po dwóch minutach (lub w przypadku straconej bramki w tym okresie) drużyna może w jego miejsce wprowadzić innego zawodnika. Boisko Boisko jest pokryte piaskiem i ma 35-37 metrów długości i 26-28 metrów szerokości, a bramki mają 2,2 metra wysokości i 5,5 metra szerokości. Bramki są umieszczone pośrodku końcowych linii boiska. Do jej poprzeczki i słupków zamontowana jest siatka. Na boisku nie ma linii wyznaczających pole karne i środek boiska, są tylko linie ograniczające boisko. Linie ograniczające boisko mają 10 cm szerokości i powinny być zrobione z taśmy w kolorze kontrastującym z p

## The RAG prompt

Context assembled from the retrieved passages, with `[document_number]` citations enforced and use of knowledge outside the context forbidden.

In [ ]:
def build_rag_prompt(query):
  rag_prompt_builder = [
      "Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.",
      "Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowiedzi na pytanie, zamiast odpowiedzi zwróć tekst: 'Nie udało mi się odnaleźć odpowiedzi na pytanie'.",
      "Numerowana lista dokumentów znajduje się poniżej",
      60 * "-",
  ]
  retrieved_chunks = search(query)
  for i, chunk in enumerate(retrieved_chunks):
    rag_prompt_builder.append(f"Dokument {i}")
    rag_prompt_builder.append(chunk)
    rag_prompt_builder.append(60 * "-")

  rag_prompt_builder.append(f"Pytanie: {query}")
  rag_prompt = "\n".join(rag_prompt_builder)
  return rag_prompt

In [ ]:
prompt = build_rag_prompt("Kim był Pan Tadeusz?")
print(prompt)

Input question: Kim był Pan Tadeusz?
top_chunks_indices_bm25=[4706, 7517, 45720, 22035, 10075, 17066, 12485, 46800, 48836, 20416, 24961, 43930, 26755, 2226, 45817]
top_chunks_indices_bi_encoder=[27335, 32326, 33892, 30227, 12219, 47089, 12644, 45790, 12449, 22158, 30141, 31065, 49569, 35897, 4224]
reranking_indices=[4224, 24961, 26755, 22158, 22035, 30227, 45720, 43930, 12449, 49569, 17066, 2226, 35897, 12219, 30141, 20416, 48836, 12485, 32326, 27335, 46800, 31065, 10075, 7517, 45790, 4706, 33892, 12644, 47089, 45817]
[48836, 7517, 46800]
Odpowiedz na pytanie użytkownika wykorzystując tylko informacje znajdujące się w dokumentach, a nie wcześniejszą wiedzę.
Udziel wysokiej jakości, poprawnej gramatycznie odpowiedzi w języku polskim. Odpowiedź powinna zawierać cytowania do dokumentów, z których pochodzą informacje. Zacytuj dokument za pomocą symbolu [nr_dokumentu] powołując się na fragment np. [0] dla fragmentu z dokumentu 0. Jeżeli w dokumentach nie ma informacji potrzebnych do odpowie

## Generator

`Llama-PLLuM-8B-instruct` in fp16 as the final component of the pipeline.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "CYFRAGOVPL/Llama-PLLuM-8B-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16, low_cpu_mem_usage=True)


def generate_with_rag(query):
  user_prompt = build_rag_prompt(query)
  messages = [
      {"role": "system", "content": "Jesteś asystentem RAG. Musisz odpowiadać w języku polskim."},
      {"role": "user", "content": user_prompt}
  ]
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt", add_generation_prompt=True, return_dict=True).to(model.device)
  print(inputs)

  model_output = model.generate(**inputs, max_new_tokens=256, pad_token_id=tokenizer.eos_token_id)
  print(model_output)

  input_length = inputs["input_ids"].shape[1]
  new_tokens = model_output[:, input_length:]

  decoded_response = tokenizer.decode(
      new_tokens[0],
      skip_special_tokens=True
  )
  return decoded_response

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
 print(generate_with_rag("Co to python?"))

Input question: Co to python?
top_chunks_indices_bm25=[6758, 49117, 12013, 18773, 45798, 15812, 26215, 30941, 12834, 7309, 21684, 26341, 7308, 47218, 7889]
top_chunks_indices_bi_encoder=[5345, 17048, 26215, 29778, 36352, 4440, 7309, 2485, 19769, 22554, 18773, 35192, 11789, 30119, 9496]
reranking_indices=[36352, 7308, 7309, 11789, 17048, 9496, 22554, 12834, 30119, 21684, 2485, 19769, 15812, 7889, 29778, 18773, 4440, 30941, 49117, 5345, 26341, 45798, 6758, 26215, 12013, 47218, 35192]
[47218, 22554, 15812]
{'input_ids': tensor([[128000, 128256,     41,  ...,  10344,     30, 128257]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1]])}
